 NOTEBOOK 04: DEPLOYMENT ARTIFACT REBUILDER

In [ ]:
import pandas as pd
import joblib

print("=" * 70)
print("LOADING EXISTING ARTIFACTS")
print("=" * 70)

LOADING EXISTING ARTIFACTS


In [ ]:
# --------------------------------------------------
# Load Dataset
# --------------------------------------------------

df = pd.read_csv("processed_eda_checkpoint.csv")

print(f"Rows Loaded: {len(df):,}")

Rows Loaded: 298,450


In [ ]:
# --------------------------------------------------
# Load Existing Model Package
# --------------------------------------------------

existing_package = joblib.load("model_v1.pkl")

print("\nLoaded model_v1.pkl successfully")


Loaded model_v1.pkl successfully


In [ ]:
# --------------------------------------------------
# Extract Existing Model
# --------------------------------------------------

if isinstance(existing_package, dict):

    if "model_binary" in existing_package:
        model_binary = existing_package["model_binary"]
    elif "model" in existing_package:
        model_binary = existing_package["model"]
    elif "classifier" in existing_package:
        model_binary = existing_package["classifier"]
    else:
        model_binary = next(
            (
                v
                for v in existing_package.values()
                if hasattr(v, "predict")
            ),
            None
        )

else:
    model_binary = existing_package

if model_binary is None:
    raise ValueError(
        "Could not locate trained model inside model_v1.pkl"
    )

print("Model extracted successfully")

# --------------------------------------------------
# Load Scaler
# --------------------------------------------------

scaler_transform = joblib.load(
    "preprocessor_scaler.pkl"
)

print("Scaler loaded successfully")

Model extracted successfully
Scaler loaded successfully


In [ ]:
# --------------------------------------------------
# Rebuild Frequency Maps
# --------------------------------------------------

print("\nReconstructing encoding maps...")

station_frequency_map = (
    df["police_station"]
    .fillna("UNKNOWN_STATION")
    .value_counts(normalize=True)
    .to_dict()
)

junction_frequency_map = (
    df["junction_name"]
    .fillna("UNKNOWN_JUNCTION")
    .value_counts(normalize=True)
    .to_dict()
)

print(
    f"Police Station Categories: "
    f"{len(station_frequency_map)}"
)

print(
    f"Junction Categories: "
    f"{len(junction_frequency_map)}"
)


Reconstructing encoding maps...
Police Station Categories: 55
Junction Categories: 170


In [ ]:
# --------------------------------------------------
# Safe Defaults
# --------------------------------------------------

default_station_frequency = min(
    station_frequency_map.values()
)

default_junction_frequency = min(
    junction_frequency_map.values()
)

In [ ]:
# --------------------------------------------------
# Canonical Feature Order
# --------------------------------------------------

features_list = [
    "latitude",
    "longitude",
    "police_station_encoded",
    "junction_encoded",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos"
]


In [ ]:
# --------------------------------------------------
# Build New Deployment Package
# --------------------------------------------------

deployment_package = {
    "model_binary": model_binary,
    "scaler_transform": scaler_transform,

    "station_frequency_map":
        station_frequency_map,

    "junction_frequency_map":
        junction_frequency_map,

    "default_station_frequency":
        default_station_frequency,

    "default_junction_frequency":
        default_junction_frequency,

    "features_list":
        features_list,

    "version":
        "2.0.0",

    "deployment_mode":
        "fully_self_contained"
}

In [ ]:
# --------------------------------------------------
# Save New Artifact
# --------------------------------------------------

joblib.dump(
    deployment_package,
    "model_v2.pkl"
)

print("\n[SUCCESS]")
print("Generated: model_v2.pkl")

print("\nContents:")
for k in deployment_package.keys():
    print(f" - {k}")


[SUCCESS]
Generated: model_v2.pkl

Contents:
 - model_binary
 - scaler_transform
 - station_frequency_map
 - junction_frequency_map
 - default_station_frequency
 - default_junction_frequency
 - features_list
 - version
 - deployment_mode
